In [0]:
dbutils.widgets.text("env", "")
env = dbutils.widgets.get("env")

#read config
from pyspark.sql.functions import col

config = (
    spark.read.table("workspace.default.config_environment")
         .filter(col("env") == env)
         .collect()[0]
)

catalog = config["catalog"]
schema = config["schema"]

from pyspark.sql.functions import *

df_customers = spark.read.table(f"{catalog}.{schema}.bronze_customers")

#display(df_customers)
silver_customers = (
    df_customers
    .withColumn(
        "city",
        when(col("city").isNull(), "UNKNOWN")
        .otherwise(col("city"))
    )
)
# display(silver_customers)


#write to table
(
    silver_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.silver_customers")
)
# display(
#     spark.read.table(f"{catalog}.{schema}.silver_customers")
# )

In [0]:
df_products = spark.read.table(f"{catalog}.{schema}.bronze_products")

from pyspark.sql.functions import trim, initcap

silver_products = (
    df_products
    .withColumn(
        "product_name",
        trim("product_name")
    )
    .withColumn(
        "category",
        initcap(trim("category"))
    )
)

# display(silver_products)

(
    silver_products
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.silver_products")
)
# display(
#     spark.read.table("workspace.default.silver_products")
# )

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import count

df_orders = spark.read.table(f"{catalog}.{schema}.bronze_orders")

#display(df_orders)

print(
    "Null order_id:",
    df_orders.filter(col("order_id").isNull()).count()
)

print(
    "Null customer_id:",
    df_orders.filter(col("customer_id").isNull()).count()
)

print(
    "Null product_id:",
    df_orders.filter(col("product_id").isNull()).count()
)

(
    df_orders
    .groupBy("order_id")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1)
    .show()
)

silver_orders = df_orders

#write data
(
    silver_orders
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.silver_orders")
)
# display(
#     spark.read.table("workspace.default.silver_orders")
# )

In [0]:
customers = spark.read.table(
    f"{catalog}.{schema}.silver_customers"
)

products = spark.read.table(
    f"{catalog}.{schema}.silver_products"
)

orders = spark.read.table(
    f"{catalog}.{schema}.silver_orders"
)

sales_silver = (
    orders
    .join(customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .select(
        "order_id",
        "order_date",
        "customer_id",
        "customer_name",
        "signup_date",
        "city",
        "state",
        "product_id",
        "product_name",
        "brand",
        "category",
        "quantity",
        "amount",
        "order_status",
        "payment_mode"
    )
)
# display(sales_silver)

# Persist as Delta
(
    sales_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.sales_silver")
)
# display(
#     spark.read.table("workspace.default.sales_silver")
# )